In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import cv2
from PIL import Image, ImageTk
import numpy as np

class AppPCD:
    def __init__(self, root):
        self.root = root
        self.root.title("Buah Vision Dashboard")
        self.root.geometry("1300x760")
        self.root.configure(bg="#1E1E2E")  # Tema dark mode modern

        # =========================================================================
        # 1. SIDEBAR PANEL (Kiri): Hanya untuk Tombol Preprocessing
        # =========================================================================
        sidebar = tk.Frame(root, bg="#252538", width=260)
        sidebar.pack(side="left", fill="y")
        sidebar.pack_propagate(False)

        # Judul Aplikasi
        tk.Label(
            sidebar,
            text="FRUIT VISION\nANALYZER",
            bg="#252538",
            fg="#F8F8F2",
            font=("Segoe UI", 16, "bold"),
            justify="center"
        ).pack(pady=25)

        tk.Frame(sidebar, bg="#44475A", height=2).pack(fill="x", padx=20, pady=10)
        tk.Label(sidebar, text="IMAGE PREPROCESSING", bg="#252538", fg="#6272A4", font=("Segoe UI", 9, "bold")).pack(anchor="w", padx=20, pady=5)

        # Daftar Tombol Preprocessing
        tombol_prep = [
            ("Load Image", self.load_image, "#50FA7B", "#282A36"),
            ("Blur Filter", self.p1_blur, "#BD93F9", "#282A36"),
            ("Sharpen Filter", self.p2_sharpen, "#BD93F9", "#282A36"),
            ("Edge Detection", self.p3_edge, "#BD93F9", "#282A36"),
            ("Denoise Filter", self.p4_denoise, "#BD93F9", "#282A36"),
            ("Opening Morph", self.p5_morfologi, "#BD93F9", "#282A36"),
            ("Adaptive Thresh", self.p6_adaptive, "#BD93F9", "#282A36"),
            ("Equalize Hist", self.p7_hist, "#BD93F9", "#282A36")
        ]

        for txt, cmd, bg_color, fg_color in tombol_prep:
            tk.Button(
                sidebar, text=txt, command=cmd, bg=bg_color, fg=fg_color,
                font=("Segoe UI", 10, "bold"), bd=0, height=2, cursor="hand2", activebackground="#FF79C6"
            ).pack(fill="x", padx=20, pady=4)

        # =========================================================================
        # 2. MAIN WORKSPACE (Kanan): Tempat Tampilan Visual Citra & Tombol Bawah
        # =========================================================================
        main_content = tk.Frame(root, bg="#1E1E2E")
        main_content.pack(side="right", expand=True, fill="both")

        # Kontainer Gambar
        display_container = tk.Frame(main_content, bg="#1E1E2E")
        display_container.pack(expand=True, fill="both", padx=30, pady=(20, 10))
        display_container.grid_columnconfigure(0, weight=1)
        display_container.grid_columnconfigure(1, weight=1)
        display_container.grid_rowconfigure(0, weight=1)

        # Panel Kiri - Citra Asli
        left_card = tk.LabelFrame(
            display_container, text=" ORIGINAL IMAGE ", bg="#282A36", fg="#F8F8F2",
            font=("Segoe UI", 12, "bold"), bd=2, relief="groove"
        )
        left_card.grid(row=0, column=0, padx=15, pady=15, sticky="nsew")
        
        self.panel_asli = tk.Label(left_card, bg="#1E1E2E")
        self.panel_asli.pack(expand=True, fill="both", padx=10, pady=10)

        # Panel Kanan - Citra Hasil
        right_card = tk.LabelFrame(
            display_container, text=" PROCESSED RESULT ", bg="#282A36", fg="#F8F8F2",
            font=("Segoe UI", 12, "bold"), bd=2, relief="groove"
        )
        right_card.grid(row=0, column=1, padx=15, pady=15, sticky="nsew")

        self.panel_hasil = tk.Label(right_card, bg="#1E1E2E")
        self.panel_hasil.pack(expand=True, fill="both", padx=10, pady=10)

        # =========================================================================
        # TATA LETAK FITUR UTAMA (Di bawah gambar, sesuai layout asli Anda)
        # =========================================================================
        bottom_buttons = tk.Frame(main_content, bg="#282A36", pady=15)
        bottom_buttons.pack(fill="x", side="top", padx=45, pady=(0, 20))

        btn_container = tk.Frame(bottom_buttons, bg="#282A36")
        btn_container.pack(anchor="center")

        tk.Button(
            btn_container, text="Segmentasi", command=self.apply_segmentation,
            bg="#27AE60", fg="white", width=22, height=2, font=("Segoe UI", 11, "bold"), bd=0, cursor="hand2"
        ).pack(side="left", padx=20)

        tk.Button(
            btn_container, text="Ekstraksi Fitur", command=self.extract_features,
            bg="#E67E22", fg="white", width=22, height=2, font=("Segoe UI", 11, "bold"), bd=0, cursor="hand2"
        ).pack(side="left", padx=20)

        tk.Button(
            btn_container, text="Klasifikasi", command=self.classify_fruit,
            bg="#C0392B", fg="white", width=22, height=2, font=("Segoe UI", 11, "bold"), bd=0, cursor="hand2"
        ).pack(side="left", padx=20)

        # =========================================================================
        # 3. STATUS BAR (Paling Bawah)
        # =========================================================================
        status_frame = tk.Frame(main_content, bg="#44475A", height=30)
        status_frame.pack(fill="x", side="bottom")

        self.status = tk.Label(
            status_frame, text="Status : Dashboard siap. Silakan muat gambar...",
            bg="#44475A", fg="#F8F8F2", font=("Segoe UI", 10, "italic"), anchor="w", padx=15
        )
        self.status.pack(fill="x", pady=4)

        # Inisialisasi variabel gambar & fitur
        self.img = None
        self.processed_img = None
        self.binary = None
        self.circularity = None  
        self.mean_r = 0
        self.mean_g = 0

    # =========================================================================
    # FUNGSI LOGIKA & PENGOLAHAN CITRA
    # =========================================================================
    def display(self, img, is_original=False):
        if img is None: return
        
        if len(img.shape) == 3:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            
        img_pil = Image.fromarray(img_rgb).resize((460, 400))
        img_tk = ImageTk.PhotoImage(img_pil)
        
        if is_original:
            self.panel_asli.config(image=img_tk, text="")
            self.panel_asli.image = img_tk
        else:
            self.panel_hasil.config(image=img_tk, text="")
            self.panel_hasil.image = img_tk

    def load_image(self):
        path = filedialog.askopenfilename(filetypes=[("Image Files", "*.jpg *.jpeg *.png *.bmp")])
        if path:
            self.img = cv2.imread(path)
            self.processed_img = self.img.copy()
            self.display(self.img, is_original=True)
            self.display(self.img, is_original=False)
            
            self.circularity = None
            self.mean_r = 0
            self.mean_g = 0
            
            self.status.config(text="Status : Gambar baru berhasil dimuat.")

    def p1_blur(self):
        if self.img is not None:
            self.processed_img = cv2.GaussianBlur(self.img, (5, 5), 0)
            self.display(self.processed_img)
            self.status.config(text="Status : Gaussian Blur diterapkan.")

    def p2_sharpen(self):
        if self.img is not None:
            kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
            self.processed_img = cv2.filter2D(self.img, -1, kernel)
            self.display(self.processed_img)
            self.status.config(text="Status : Image Sharpening diterapkan.")

    def p3_edge(self):
        if self.img is not None:
            gray = cv2.cvtColor(self.img, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 200)
            self.processed_img = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
            self.display(self.processed_img)
            self.status.config(text="Status : Deteksi Tepi Canny diterapkan.")

    def p4_denoise(self):
        if self.img is not None:
            self.processed_img = cv2.medianBlur(self.img, 5)
            self.display(self.processed_img)
            self.status.config(text="Status : Median Denoise diterapkan.")

    def p5_morfologi(self):
        if self.img is not None:
            kernel = np.ones((5, 5), np.uint8)
            self.processed_img = cv2.morphologyEx(self.img, cv2.MORPH_OPEN, kernel)
            self.display(self.processed_img)
            self.status.config(text="Status : Operasi Morfologi (Opening) diterapkan.")

    def p6_adaptive(self):
        if self.img is not None:
            gray = cv2.cvtColor(self.img, cv2.COLOR_BGR2GRAY)
            thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
            self.processed_img = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
            self.display(self.processed_img)
            self.status.config(text="Status : Adaptive Thresholding diterapkan.")

    def p7_hist(self):
        if self.img is not None:
            img_yuv = cv2.cvtColor(self.img, cv2.COLOR_BGR2YUV)
            img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
            self.processed_img = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)
            self.display(self.processed_img)
            self.status.config(text="Status : Ekualisasi Histogram diterapkan.")

    def apply_segmentation(self):
        if self.processed_img is not None:
            if len(self.processed_img.shape) == 3:
                gray = cv2.cvtColor(self.processed_img, cv2.COLOR_BGR2GRAY)
            else:
                gray = self.processed_img
            
            _, self.binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            
            if self.binary[0, 0] == 255:
                self.binary = cv2.bitwise_not(self.binary)
                
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
            self.binary = cv2.morphologyEx(self.binary, cv2.MORPH_CLOSE, kernel)

            self.display(cv2.cvtColor(self.binary, cv2.COLOR_GRAY2BGR))
            self.status.config(text="Status : Segmentasi & Morfologi Closing selesai.")
        else:
            messagebox.showwarning("Error", "Pilih atau proses gambar terlebih dahulu!")

    def extract_features(self):
        if self.binary is not None and self.img is not None:
            contours, _ = cv2.findContours(self.binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                c = max(contours, key=cv2.contourArea)
                self.luas = cv2.contourArea(c)
                perimeter = cv2.arcLength(c, True)
                
                self.circularity = (4 * np.pi * self.luas) / (perimeter ** 2) if perimeter > 0 else 0
                
                mask = np.zeros(self.binary.shape, dtype=np.uint8)
                cv2.drawContours(mask, [c], -1, 255, -1)
                
                mean_val = cv2.mean(self.img, mask=mask)
                self.mean_g = mean_val[1]  
                self.mean_r = mean_val[2]  
                
                info_text = (f"Luas: {self.luas:.2f}\n"
                             f"Circularity: {self.circularity:.4f}\n"
                             f"Rata-rata R (Merah): {self.mean_r:.1f}\n"
                             f"Rata-rata G (Hijau): {self.mean_g:.1f}")
                             
                messagebox.showinfo("Fitur Ekstraksi", info_text)
                self.status.config(text="Status : Fitur Bentuk & Warna berhasil diekstraksi.")
            else:
                messagebox.showwarning("Error", "Objek tidak ditemukan pada gambar biner!")
        else:
            messagebox.showwarning("Error", "Lakukan proses Segmentasi terlebih dahulu!")

    def classify_fruit(self):
        if self.circularity is not None:
            if self.circularity > 0.60:
                if self.mean_r > 140 and self.mean_g > 90:
                    hasil = "Jeruk (Bulat Oranye/Kuning)"
                elif self.mean_r > 120 and self.mean_g <= 90:
                    hasil = "Apel (Bulat Merah)"
                elif self.mean_g > self.mean_r and self.mean_g > 115:
                    hasil = "Apel Hijau (Bulat)"
                else:
                    hasil = "Mangga / Buah Bulat Menengah"
                    
            elif 0.35 < self.circularity <= 0.60:
                if self.mean_g > self.mean_r:
                    hasil = "Mangga (Lonjong/Oval)"
                else:
                    hasil = "Apel (Oval Merah) / Buah Lain"
                    
            elif 0.05 < self.circularity <= 0.35:
                hasil = "Pisang (Lonjong Kuning)"
            else:
                hasil = "Tidak Terdeteksi"
                
            messagebox.showinfo("Hasil Klasifikasi", f"Terdeteksi sebagai: {hasil}")
            self.status.config(text=f"Status : Klasifikasi Selesai ({hasil})")
        else:
            messagebox.showwarning("Error", "Ekstraksi fitur belum dilakukan! Tekan tombol 'Ekstraksi Fitur' terlebih dahulu.")

if __name__ == "__main__":
    root = tk.Tk()
    app = AppPCD(root)
    root.mainloop()